# Classer sans exemple et avec quelques exemples : CLIP

CLIP a appris, sur 400 millions de paires image-légende du web, à placer une image et sa légende au même endroit d'un espace de vecteurs. Dans ce TP, vous utilisez cet espace pour classer des photos de paysages sans aucun exemple étiqueté (*zero-shot*) : une classe y est décrite par un texte. Vous mesurez l'effet de ce texte, calculez la perte contrastive avec laquelle CLIP a été entraîné, et cherchez des images par une phrase. Vous passez ensuite au *few-shot* : quelques photos étiquetées par classe, résumées par des prototypes ou par une régression logistique, sur les paysages puis sur des circuits imprimés dont il faut reconnaître les défauts.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Le TP tourne sans GPU : le calcul des vecteurs des images prend une à deux minutes par jeu.

## Installation et données

Deux jeux de [shuuchuu/datasets](https://github.com/shuuchuu/datasets) :

- la partie test du jeu *Intel Image Classification* (`landscape/seg_test`) : 3 000 photos de 150 × 150 pixels, en six classes (`buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`), 50 Mo. Le TP en tire 150 par classe ;
- les circuits imprimés `pcb4` de VisA (`visa-subset/pcb4`, Amazon, CC BY 4.0) : 351 photos de cartes sans défaut et 98 de cartes avec un défaut (composant manquant, en trop, mal placé, abîmé, brûlure, rayure), 12 Mo.

Le modèle est [`openai/clip-vit-base-patch32`](https://huggingface.co/openai/clip-vit-base-patch32) (151 millions de paramètres, 600 Mo), téléchargé à son premier chargement. Le code suit l'API de transformers 5, que la première ligne installe si Colab a une version plus ancienne.

In [ ]:
!pip install -q "transformers>=5"
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape/seg_test
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh visa-subset/pcb4

In [ ]:
import math
import pathlib
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, roc_auc_score
from transformers import CLIPModel, CLIPProcessor

torch.set_grad_enabled(False)  # aucun entraînement dans ce TP
nom = "openai/clip-vit-base-patch32"
modele = CLIPModel.from_pretrained(nom).eval()
proc = CLIPProcessor.from_pretrained(nom)

`chemins` et `y` décrivent l'échantillon de paysages : 150 photos par classe, tirées au hasard avec une graine fixe, et leur classe (un entier de 0 à 5, l'indice dans `classes`). `sujets` nomme les mêmes classes avec leur article, pour les phrases en anglais (« a photo of a forest. ») ; `classes` sert aux titres. `chemins_pcb` et `y_pcb` font de même pour les circuits imprimés, qui ne servent qu'à l'exercice 9. `afficher` montre des photos côte à côte.

In [ ]:
classes = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
sujets = ["buildings", "a forest", "a glacier", "a mountain", "the sea", "a street"]
rng = np.random.default_rng(0)
chemins, y = [], []
for i, classe in enumerate(classes):
  fichiers = sorted(pathlib.Path("seg_test", classe).glob("*.jpg"))
  for j in sorted(rng.choice(len(fichiers), 150, replace=False)):
    chemins.append(fichiers[j])
    y.append(i)
y = np.array(y)


def afficher(chemins, titres=None, taille=2.5):
  fig, axes = plt.subplots(1, len(chemins), figsize=(taille * len(chemins), taille))
  for i, (ax, chemin) in enumerate(zip(np.atleast_1d(axes), chemins)):
    ax.imshow(Image.open(chemin))
    if titres:
      ax.set_title(titres[i])
    ax.axis("off")
  plt.show()


print(len(chemins), np.bincount(y))
afficher([chemins[np.flatnonzero(y == i)[0]] for i in range(6)], titres=classes)

# Les circuits imprimés de l'exercice 9 : 0 sans défaut, 1 avec un défaut.
sains = sorted(pathlib.Path("pcb4").glob("*/good/*.jpg"))
defauts = sorted(
  p for p in pathlib.Path("pcb4", "test").glob("*/*.jpg") if p.parent.name != "good"
)
chemins_pcb = sains + defauts
y_pcb = np.array([0] * len(sains) + [1] * len(defauts))
print(len(sains), len(defauts))
afficher(
  [sains[0], defauts[0], defauts[40]],
  titres=["sans défaut", defauts[0].parent.name, defauts[40].parent.name],
)

## 1. Une image et six textes dans le même espace

Encodez avec l'encodeur d'images de CLIP trois photos de l'échantillon : la première forêt (`chemins[150]`), la deuxième (`chemins[151]`) et la première rue (`chemins[750]`) ; et avec son encodeur de texte les six textes `"a photo of {sujet}."` (« a photo of a forest. »). Normalisez chaque vecteur (norme 1) : leur produit scalaire est alors leur similarité cosinus. Affichez la taille des vecteurs, les six similarités de la première forêt avec les textes, puis les probabilités que CLIP donne à chaque classe : le softmax des similarités multipliées par `modele.logit_scale.exp()`. Affichez enfin la similarité de la première forêt avec chacune des deux autres photos.

Les similarités aux textes sont-elles très différentes d'une classe à l'autre ? Et les probabilités ? Comparez les similarités entre deux photos à celle de la forêt et de son texte : peut-on mettre dans un même classifieur des vecteurs d'images et de textes, et prendre le plus proche ?

**Documentation utile**

- https://huggingface.co/docs/transformers/model_doc/clip#transformers.CLIPModel.get_image_features
- https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `proc(images=images, return_tensors="pt")` prépare une liste d'images (redimensionnement en 224 × 224, normalisation) ; `modele.get_image_features(**entrees)` renvoie un objet dont `.pooler_output` contient les vecteurs projetés dans l'espace commun, de taille `(3, 512)` pour trois images.
- `proc(text=textes, return_tensors="pt", padding=True)` découpe les textes en jetons, complétés à la même longueur ; `modele.get_text_features(**entrees).pooler_output` donne un vecteur par texte.
- `F.normalize(v, dim=-1)` divise chaque ligne par sa norme. `v @ t.T` donne alors la matrice des similarités cosinus, `v[0] @ v[1]` celle de deux images.
- `modele.logit_scale.exp()` vaut 100 : l'inverse de la température apprise pendant l'entraînement.

```python
images = [Image.open(chemins[i]).convert("RGB") for i in (150, 151, 750)]
v = modele.get_image_features(**proc(images=images, return_tensors="pt")).pooler_output
v = F.normalize(v, dim=-1)
textes = [f"a photo of {s}." for s in sujets]
t = ...
t = F.normalize(t, dim=-1)
similarites = v[:1] @ t.T
probas = (...).softmax(dim=-1)
print(...)
```

### Solution

In [ ]:
images = [Image.open(chemins[i]).convert("RGB") for i in (150, 151, 750)]
v = modele.get_image_features(**proc(images=images, return_tensors="pt")).pooler_output
v = F.normalize(v, dim=-1)
textes = [f"a photo of {s}." for s in sujets]
t = modele.get_text_features(
  **proc(text=textes, return_tensors="pt", padding=True)
).pooler_output
t = F.normalize(t, dim=-1)
print(v.shape, t.shape)
similarites = v[:1] @ t.T
probas = (modele.logit_scale.exp() * similarites).softmax(dim=-1)
for classe, s, p in zip(classes, similarites[0], probas[0]):
  print(f"{classe:10} similarité {s:.3f}   probabilité {p:.3f}")
print(f"Forêt et autre forêt : {v[0] @ v[1]:.3f}   forêt et rue : {v[0] @ v[2]:.3f}")
afficher([chemins[i] for i in (150, 151, 750)], titres=["forêt", "forêt", "rue"])
# Les similarités cosinus d'une image et d'un texte sont faibles (autour de 0,2 à
# 0,3) et proches les unes des autres : quelques centièmes séparent la bonne classe
# des autres. Multipliées par 100 avant le softmax, ces petits écarts deviennent une
# probabilité proche de 1 pour la classe la plus similaire.
# Entre deux photos, la similarité est bien plus haute : autour de 0,85 pour les deux
# forêts, 0,7 encore pour la forêt et la rue, contre 0,3 pour la forêt et son texte.
# Les vecteurs d'images et ceux des textes occupent deux régions distinctes de
# l'espace (le *modality gap*) : mêlés dans un classement au plus proche, la rue
# battrait le texte « a photo of a forest. ». On ne les mélange pas sans les recaler ;
# on compare une image à des textes, ou à d'autres images.

## 2. Les vecteurs de l'échantillon

Écrivez deux fonctions, réutilisées dans toute la suite :

- `vecteurs_images(chemins, taille_lot=32)` : les vecteurs normalisés de toutes les images de `chemins`, calculés par lots de `taille_lot` images, en un tenseur de taille `(len(chemins), 512)` ;
- `vecteurs_textes(textes)` : les vecteurs normalisés d'une liste de textes.

Calculez `V`, les vecteurs des 900 photos de l'échantillon, et mesurez la durée du calcul. Vérifiez sur la photo de l'exercice 1 que `V[150]` est bien le vecteur `v` trouvé alors.

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Un lot : `[Image.open(c).convert("RGB") for c in chemins[i : i + taille_lot]]`, pour `i` de 0 à `len(chemins)` par pas de `taille_lot`.
- `torch.cat(liste, dim=0)` met bout à bout les vecteurs des lots.
- `torch.allclose(a, b, atol=1e-5)` compare deux tenseurs aux erreurs d'arrondi près.

```python
def vecteurs_images(chemins, taille_lot=32):
  lots = []
  for i in range(0, len(chemins), taille_lot):
    images = ...
    sortie = modele.get_image_features(**proc(images=images, return_tensors="pt"))
    lots.append(sortie.pooler_output)
  return F.normalize(torch.cat(lots), dim=-1)


def vecteurs_textes(textes): ...
```

### Solution

In [ ]:
def vecteurs_images(chemins, taille_lot=32):
  lots = []
  for i in range(0, len(chemins), taille_lot):
    images = [Image.open(c).convert("RGB") for c in chemins[i : i + taille_lot]]
    sortie = modele.get_image_features(**proc(images=images, return_tensors="pt"))
    lots.append(sortie.pooler_output)
  return F.normalize(torch.cat(lots), dim=-1)


def vecteurs_textes(textes):
  entrees = proc(text=textes, return_tensors="pt", padding=True)
  return F.normalize(modele.get_text_features(**entrees).pooler_output, dim=-1)


debut = time.perf_counter()
V = vecteurs_images(chemins)
print(V.shape, f"{time.perf_counter() - debut:.0f} s")
print(torch.allclose(V[150], v[0], atol=1e-5))
# Sans GPU, une centaine de millisecondes par image : c'est l'étape la plus lente du
# TP, faite une seule fois. Tout le reste ne manipule que ces 900 vecteurs.

## 3. Classer sans exemple

Classez les 900 photos sans aucun exemple étiqueté : la classe prédite d'une photo est celle du texte `"a photo of {sujet}."` dont son vecteur est le plus proche. Calculez l'exactitude, puis affichez la matrice de confusion avec les noms des classes.

Quelles classes CLIP confond-il ? Affichez cinq photos mal classées de la classe la plus confondue, avec la classe prédite en titre : l'étiquette du jeu est-elle toujours la bonne ?

**Documentation utile**

- https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html
- https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `V @ T.T` donne une matrice `(900, 6)` de similarités ; `.argmax(dim=1)` la classe la plus proche de chaque photo (`.numpy()` en fait un tableau NumPy, `pred_texte`). Le softmax ne change pas l'ordre : inutile ici.
- `ConfusionMatrixDisplay.from_predictions(y, pred_texte, display_labels=classes)` trace la matrice, les vraies classes en lignes.
- `np.bincount(y[pred_texte != y], minlength=6)` compte les erreurs de chaque vraie classe ; `.argmax()` donne la plus confondue, `cible`.
- `np.flatnonzero((y == cible) & (pred_texte != cible))` donne les indices de ses photos mal classées.

```python
T = vecteurs_textes([f"a photo of {s}." for s in sujets])
pred_texte = ...
print(accuracy_score(y, pred_texte))
ConfusionMatrixDisplay.from_predictions(...)
plt.show()
cible = ...
erreurs = np.flatnonzero(...)[:5]
afficher(
  [chemins[i] for i in erreurs], titres=[classes[pred_texte[i]] for i in erreurs]
)
```

### Solution

In [ ]:
T = vecteurs_textes([f"a photo of {s}." for s in sujets])
pred_texte = (V @ T.T).argmax(dim=1).numpy()
print(f"Exactitude sans exemple : {accuracy_score(y, pred_texte):.1%}")
ConfusionMatrixDisplay.from_predictions(
  y, pred_texte, display_labels=classes, xticks_rotation=45
)
plt.show()
cible = np.bincount(y[pred_texte != y], minlength=6).argmax()
print(f"Classe la plus confondue : {classes[cible]}")
erreurs = np.flatnonzero((y == cible) & (pred_texte != cible))[:5]
afficher(
  [chemins[i] for i in erreurs], titres=[classes[pred_texte[i]] for i in erreurs]
)
# Plus de 90 % d'exactitude sans une seule photo étiquetée : un SVM sur des
# caractéristiques HOG, entraîné sur 2 100 photos de ce jeu, en atteint environ 70 %.
# Les erreurs se concentrent sur glacier et mountain, et sur street et buildings :
# beaucoup de photos de glaciers montrent des montagnes enneigées, beaucoup de rues
# sont bordées d'immeubles. Plusieurs étiquettes du jeu se discutent.

## 4. Le texte compte

Comparez l'exactitude du classement sans exemple avec quatre façons de décrire les classes :

- le nom seul (`"forest"`) ;
- `"a photo of {sujet}."`, celle de l'exercice 3 ;
- une phrase en français : `"une photo de bâtiments."`, `"une photo d'une forêt."`, `"une photo d'un glacier."`, `"une photo d'une montagne."`, `"une photo de la mer."`, `"une photo d'une rue."` ;
- un ensemble de gabarits : le vecteur d'une classe est la moyenne, renormalisée, des vecteurs de plusieurs phrases, remplies avec `sujets` (`"a photo of {}."`, `"a landscape photo of {}."`, `"a blurry photo of {}."`, `"a picture of {}."`).

Écrivez une fonction `exactitude_sans_exemple(T)` qui prend la matrice `(6, 512)` des vecteurs des classes. Quelle description est la meilleure ? Que donne le français, et pourquoi ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Pour l'ensemble de gabarits : pour chaque classe, `vecteurs_textes([g.format(s) for g in gabarits]).mean(dim=0)`, puis `torch.stack(...)` sur les six classes et `F.normalize(..., dim=-1)`.
- CLIP d'OpenAI a été entraîné sur des légendes en anglais.

```python
def exactitude_sans_exemple(T):
  return accuracy_score(y, (V @ T.T).argmax(dim=1).numpy())


francais = [...]
gabarits = ["a photo of {}.", "a landscape photo of {}.", ...]
T_ensemble = F.normalize(
  torch.stack(
    [vecteurs_textes([g.format(s) for g in gabarits]).mean(dim=0) for s in sujets]
  ),
  dim=-1,
)
descriptions = {
  "nom seul": vecteurs_textes(classes),
  "a photo of …": ...,
  "en français": ...,
  "ensemble de gabarits": T_ensemble,
}
for nom_desc, T_desc in descriptions.items():
  print(nom_desc, exactitude_sans_exemple(T_desc))
```

### Solution

In [ ]:
def exactitude_sans_exemple(T):
  return accuracy_score(y, (V @ T.T).argmax(dim=1).numpy())


francais = [
  "une photo de bâtiments.",
  "une photo d'une forêt.",
  "une photo d'un glacier.",
  "une photo d'une montagne.",
  "une photo de la mer.",
  "une photo d'une rue.",
]
gabarits = [
  "a photo of {}.",
  "a landscape photo of {}.",
  "a blurry photo of {}.",
  "a picture of {}.",
]
T_ensemble = F.normalize(
  torch.stack(
    [vecteurs_textes([g.format(s) for g in gabarits]).mean(dim=0) for s in sujets]
  ),
  dim=-1,
)
descriptions = {
  "nom seul": vecteurs_textes(classes),
  "a photo of …": vecteurs_textes([f"a photo of {s}." for s in sujets]),
  "en français": vecteurs_textes(francais),
  "ensemble de gabarits": T_ensemble,
}
for nom_desc, T_desc in descriptions.items():
  print(f"{nom_desc:22} {exactitude_sans_exemple(T_desc):.1%}")
# Les descriptions en anglais se valent à deux points près, et l'ensemble de
# gabarits ne fait pas mieux ici : sur des classes aussi simples, le nom suffit.
# Le français perd plusieurs points : CLIP d'OpenAI a
# appris sur des légendes en anglais, et ne connaît du français que ce que le web
# anglophone en contient. Pour du texte en français, un modèle multilingue
# (SigLIP 2) convient mieux. Sur d'autres jeux, le gabarit et l'ensemble font gagner
# davantage : à mesurer sur ses propres données.

## 5. La perte contrastive d'un lot

CLIP a été entraîné par lots de paires image-légende : pour chaque image, retrouver sa légende parmi celles du lot, et pour chaque légende, son image. Formez un lot de six paires : la première photo de chaque classe et le texte `"a photo of {sujet}."`. Écrivez `perte_clip(V_lot, T_lot, echelle)`, qui calcule la matrice `S` des similarités multipliées par `echelle`, puis la perte contrastive symétrique (InfoNCE) : la moyenne de l'entropie croisée par lignes (image → texte) et par colonnes (texte → image), la bonne réponse de la ligne `i` étant la colonne `i`.

Avec l'échelle de CLIP, `modele.logit_scale.exp()` (100 : l'inverse de la température $\tau$ apprise), comparez la perte du lot à celle du même lot dont les légendes sont décalées d'un cran (chaque image associée à la légende de la classe suivante), et à $\log 6$, la perte d'un modèle qui répondrait au hasard. Recommencez avec les échelles 10 et 1 ($\tau$ = 0,1 et 1). Que change la température ? Pourquoi CLIP ne se contente-t-il pas de $\tau$ = 1 ?

**Documentation utile**

- https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `F.cross_entropy(S, cibles)` traite chaque ligne de `S` comme les scores d'une classification à six classes ; `cibles = torch.arange(6)` dit que la bonne réponse de la ligne `i` est `i`. `S.T` donne les colonnes.
- `torch.roll(T_lot, 1, dims=0)` décale les lignes d'un cran.
- `modele.logit_scale.exp().item()` donne l'échelle de CLIP en nombre Python.

```python
premieres = [np.flatnonzero(y == i)[0] for i in range(6)]
V_lot = V[premieres]
T_lot = vecteurs_textes([f"a photo of {s}." for s in sujets])
cibles = torch.arange(6)


def perte_clip(V_lot, T_lot, echelle):
  S = echelle * V_lot @ T_lot.T
  return (F.cross_entropy(S, cibles) + ...) / 2


decalees = torch.roll(T_lot, 1, dims=0)
for echelle in [modele.logit_scale.exp().item(), 10, 1]:
  print(...)
```

### Solution

In [ ]:
premieres = [np.flatnonzero(y == i)[0] for i in range(6)]
V_lot = V[premieres]
T_lot = vecteurs_textes([f"a photo of {s}." for s in sujets])
cibles = torch.arange(6)


def perte_clip(V_lot, T_lot, echelle):
  S = echelle * V_lot @ T_lot.T
  return (F.cross_entropy(S, cibles) + F.cross_entropy(S.T, cibles)) / 2


decalees = torch.roll(T_lot, 1, dims=0)
print(f"Au hasard : {math.log(6):.3f}")
for echelle in [modele.logit_scale.exp().item(), 10, 1]:
  print(
    f"Échelle {echelle:3.0f} : paires justes {perte_clip(V_lot, T_lot, echelle):.3f}, "
    f"légendes décalées {perte_clip(V_lot, decalees, echelle):.3f}"
  )
# À l'échelle de CLIP, la perte des paires justes est faible, bien sous log 6 :
# chaque image est plus proche de sa légende que des cinq autres. Décalées, elle
# dépasse largement log 6 : le modèle est sûr de lui et se trompe.
# Les similarités du lot tiennent toutes entre 0,18 et 0,31. Avec τ = 1, ces écarts
# de quelques centièmes donnent un softmax presque uniforme : les deux pertes restent
# près de log 6, et même des vecteurs parfaits (similarités bornées par 1) ne la
# feraient pas descendre bien bas. Une petite température amplifie les écarts : la
# perte distingue les bonnes paires des mauvaises, et les négatives les plus proches
# y pèsent le plus. CLIP apprend τ ; son inverse atteint 100, la borne que fixe
# l'entraînement. Il fait baisser cette perte sur des lots de 32 768 paires : chaque
# image y a 32 767 légendes à écarter.

## 6. Chercher des images par une phrase

Le même espace sert à chercher : le vecteur d'une phrase quelconque se compare à ceux des 900 photos. Écrivez une fonction `chercher(phrase, k=5)` qui affiche les `k` photos les plus proches de la phrase, avec leur similarité en titre. Essayez-la sur `"a snowy mountain at sunset"`, `"a boat on the sea"` et `"a crowded street at night"`, puis sur des phrases de votre choix.

Les photos trouvées correspondent-elles à la phrase entière, ou seulement à sa classe ?

**Documentation utile**

- https://docs.pytorch.org/docs/stable/generated/torch.topk.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `vecteurs_textes([phrase])[0]` donne le vecteur de la phrase ; `V @ q` ses 900 similarités.
- `torch.topk(similarites, k)` renvoie les `k` plus grandes valeurs (`.values`) et leurs indices (`.indices`).

```python
def chercher(phrase, k=5):
  q = vecteurs_textes([phrase])[0]
  meilleures = torch.topk(V @ q, k)
  afficher(
    [chemins[i] for i in meilleures.indices],
    titres=[...],
  )
```

### Solution

In [ ]:
def chercher(phrase, k=5):
  q = vecteurs_textes([phrase])[0]
  meilleures = torch.topk(V @ q, k)
  print(phrase)
  afficher(
    [chemins[i] for i in meilleures.indices],
    titres=[f"{s:.3f}" for s in meilleures.values],
  )


for phrase in [
  "a snowy mountain at sunset",
  "a boat on the sea",
  "a crowded street at night",
]:
  chercher(phrase)
# Les photos trouvées vont au-delà de la classe : couleurs du couchant, bateaux,
# rues éclairées la nuit. Aucune étiquette ne décrit ces détails : c'est l'apprentissage sur des
# légendes libres qui les rend cherchables. Les similarités restent faibles (autour de
# 0,3), comme à l'exercice 1 : seul leur ordre compte.

## 7. Quelques exemples par classe : les prototypes

Supposons maintenant qu'on ait quelques photos étiquetées par classe, et aucun texte. La méthode la plus simple, celle des réseaux prototypiques, résume chaque classe par un prototype : la moyenne, renormalisée, des vecteurs de ses exemples ; une photo prend la classe du prototype le plus proche.

Écrivez `tirer(k, graine)`, qui tire au hasard `k` indices de photos par classe et renvoie ces indices (le support) et ceux de toutes les autres photos (l'évaluation), `exactitude_prototypes(k, graine)`, qui calcule les prototypes sur le support et l'exactitude sur l'évaluation, et `exactitude_texte(k, graine)`, l'exactitude du classement sans exemple de l'exercice 3 sur les mêmes photos d'évaluation : deux méthodes ne se comparent que sur les mêmes photos. Pour `k` = 1, 2, 4, 8 et 16, affichez la moyenne et l'écart type de l'exactitude des prototypes sur 20 tirages, et la moyenne de celle du classement sans exemple.

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `np.random.default_rng(graine).choice(np.flatnonzero(y == c), k, replace=False)` tire `k` photos de la classe `c`.
- `np.setdiff1d(np.arange(len(y)), support)` donne les indices qui ne sont pas dans le support.
- Le prototype de la classe `c` : `V[support][y[support] == c].mean(dim=0)`.
- Les prédictions sans exemple de l'exercice 3 sont dans `pred_texte` : il suffit de les comparer à `y` sur les photos d'évaluation.

```python
def tirer(k, graine):
  rng = np.random.default_rng(graine)
  support = np.concatenate([... for c in range(6)])
  return support, np.setdiff1d(np.arange(len(y)), support)


def exactitude_prototypes(k, graine):
  support, evaluation = tirer(k, graine)
  P = F.normalize(torch.stack([... for c in range(6)]), dim=-1)
  pred = (V[evaluation] @ P.T).argmax(dim=1).numpy()
  return accuracy_score(y[evaluation], pred)


def exactitude_texte(k, graine):
  evaluation = tirer(k, graine)[1]
  return ...


for k in [1, 2, 4, 8, 16]:
  scores = [exactitude_prototypes(k, graine) for graine in range(20)]
  sans_exemple = ...
  print(...)
```

### Solution

In [ ]:
def tirer(k, graine):
  rng = np.random.default_rng(graine)
  support = np.concatenate(
    [rng.choice(np.flatnonzero(y == c), k, replace=False) for c in range(6)]
  )
  return support, np.setdiff1d(np.arange(len(y)), support)


def exactitude_prototypes(k, graine):
  support, evaluation = tirer(k, graine)
  P = F.normalize(
    torch.stack([V[support][y[support] == c].mean(dim=0) for c in range(6)]), dim=-1
  )
  pred = (V[evaluation] @ P.T).argmax(dim=1).numpy()
  return accuracy_score(y[evaluation], pred)


def exactitude_texte(k, graine):
  evaluation = tirer(k, graine)[1]
  return accuracy_score(y[evaluation], pred_texte[evaluation])


for k in [1, 2, 4, 8, 16]:
  scores = [exactitude_prototypes(k, graine) for graine in range(20)]
  sans_exemple = np.mean([exactitude_texte(k, graine) for graine in range(20)])
  print(
    f"k = {k:2} : prototypes {np.mean(scores):.1%} ± {np.std(scores):.1%}, "
    f"sans exemple {sans_exemple:.1%}"
  )
# Avec un seul exemple par classe, l'exactitude dépend beaucoup de la photo tirée
# (l'écart type est grand) ; elle monte avec k, mais même 16 photos par classe
# restent sous le classement sans exemple, mesuré sur les mêmes photos. Ces classes sont nommées dans des
# millions de légendes du web : le texte les décrit mieux que quelques photos.

## 8. Une régression logistique sur les vecteurs

Au lieu de prototypes, entraînez une régression logistique sur les vecteurs du support (on parle de *linear probe* : un classifieur linéaire sur un encodeur gelé). Écrivez `exactitude_regression(k, graine)` sur le modèle de `exactitude_prototypes`, avec les mêmes tirages, et comparez les deux méthodes pour `k` = 1, 4 et 16.

Entraînez enfin la même régression logistique sur 100 photos par classe, et évaluez-la sur les 300 autres : dépasse-t-elle le classement sans exemple sur ces mêmes photos ?

**Documentation utile**

- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `LogisticRegression(max_iter=1000).fit(X, y)` entraîne la régression ; `max_iter` lui laisse assez d'itérations pour converger.
- `V[support].numpy()` convertit les vecteurs pour scikit-learn.
- `tirer(100, 0)` donne 600 photos de support et 300 d'évaluation ; `exactitude_texte(100, 0)` le classement sans exemple sur ces 300 photos.

```python
def exactitude_regression(k, graine):
  support, evaluation = tirer(k, graine)
  regression = LogisticRegression(max_iter=1000).fit(...)
  return accuracy_score(y[evaluation], regression.predict(...))
```

### Solution

In [ ]:
def exactitude_regression(k, graine):
  support, evaluation = tirer(k, graine)
  regression = LogisticRegression(max_iter=1000).fit(V[support].numpy(), y[support])
  return accuracy_score(y[evaluation], regression.predict(V[evaluation].numpy()))


for k in [1, 4, 16]:
  prototypes = np.mean([exactitude_prototypes(k, g) for g in range(20)])
  regression = np.mean([exactitude_regression(k, g) for g in range(20)])
  print(f"k = {k:2} : prototypes {prototypes:.1%}, régression {regression:.1%}")
print(
  f"100 photos par classe : régression {exactitude_regression(100, 0):.1%}, "
  f"sans exemple {exactitude_texte(100, 0):.1%}"
)
# Avec un seul exemple par classe, la régression logistique gagne quelques points sur
# les prototypes, mais reste loin du texte ; à 16 exemples, les deux méthodes se
# valent. À 100 photos par classe, la régression rejoint le classement sans exemple,
# mesuré sur les mêmes 300 photos, et ne le dépasse que d'un ou deux points selon le
# tirage ; sur les 3 000 photos du jeu, une régression entraînée sur 350 photos par
# classe ne le dépasse que d'un point. L'article de CLIP trouve qu'en moyenne, sur
# ses jeux de test, le zero-shot vaut une régression logistique entraînée sur 4
# exemples par classe : ici, il en vaut bien plus.

## 9. Quand le texte ne suffit pas : des défauts de circuits imprimés

Sur les circuits imprimés de VisA, il s'agit de reconnaître les cartes qui ont un défaut. `chemins_pcb` et `y_pcb` (1 pour un défaut) sont définis au début du TP. Calculez leurs vecteurs, puis mesurez par l'aire sous la courbe ROC (AUC, qui ne dépend pas d'un seuil, utile ici où les classes sont déséquilibrées) :

- le classement sans exemple : le score d'une carte est la probabilité du texte `"a photo of a damaged circuit board."` face à `"a photo of a flawless circuit board."` ;
- les prototypes, avec `k` = 1, 4 et 16 cartes de chaque sorte, sur 20 tirages : le score d'une carte est sa similarité au prototype des cartes défectueuses, moins sa similarité à celui des cartes saines.

Le texte suffit-il cette fois ? Pourquoi ?

**Documentation utile**

- https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `roc_auc_score(y_vrai, score)` : 0,5 pour un score au hasard, 1 pour un score qui place toutes les cartes défectueuses au-dessus des saines.
- La probabilité du défaut, sans exemple : `(modele.logit_scale.exp() * V_pcb @ T_pcb.T).softmax(dim=1)[:, 1]`, avec les textes dans l'ordre (sain, défaut).
- Le tirage du support se fait comme à l'exercice 7, avec deux classes au lieu de six.

```python
V_pcb = vecteurs_images(chemins_pcb)
T_pcb = vecteurs_textes([...])
score = ...
print(roc_auc_score(y_pcb, score))


def auc_prototypes(k, graine):
  rng = np.random.default_rng(graine)
  support = np.concatenate(
    [rng.choice(np.flatnonzero(y_pcb == c), k, replace=False) for c in (0, 1)]
  )
  evaluation = ...
  sain = F.normalize(V_pcb[support][y_pcb[support] == 0].mean(dim=0), dim=0)
  defaut = ...
  score = ...
  return roc_auc_score(y_pcb[evaluation], score)
```

### Solution

In [ ]:
V_pcb = vecteurs_images(chemins_pcb)
T_pcb = vecteurs_textes(
  ["a photo of a flawless circuit board.", "a photo of a damaged circuit board."]
)
score = (modele.logit_scale.exp() * V_pcb @ T_pcb.T).softmax(dim=1)[:, 1]
print(f"Sans exemple : AUC {roc_auc_score(y_pcb, score.numpy()):.2f}")


def auc_prototypes(k, graine):
  rng = np.random.default_rng(graine)
  support = np.concatenate(
    [rng.choice(np.flatnonzero(y_pcb == c), k, replace=False) for c in (0, 1)]
  )
  evaluation = np.setdiff1d(np.arange(len(y_pcb)), support)
  sain = F.normalize(V_pcb[support][y_pcb[support] == 0].mean(dim=0), dim=0)
  defaut = F.normalize(V_pcb[support][y_pcb[support] == 1].mean(dim=0), dim=0)
  score = V_pcb[evaluation] @ defaut - V_pcb[evaluation] @ sain
  return roc_auc_score(y_pcb[evaluation], score.numpy())


for k in [1, 4, 16]:
  aucs = [auc_prototypes(k, graine) for graine in range(20)]
  print(f"k = {k:2} : AUC {np.mean(aucs):.2f} ± {np.std(aucs):.2f}")
# Sans exemple, l'AUC reste faible (0,65) : un composant manquant ou une petite
# brûlure sur cette carte précise ne ressemble à rien de ce que décrit « damaged
# circuit board » dans les légendes du web. Quelques cartes étiquetées montrent ce
# qu'est un défaut ici, et l'AUC monte nettement avec k. Commencer par le zero-shot,
# le mesurer, et étiqueter quelques exemples là où il échoue : domaines spécialisés,
# défauts, classes qu'aucune légende ne nomme.